In [ ]:
%reload_ext autoreload
%autoreload 2

import warnings
warnings.filterwarnings('ignore')

from sklearn.utils import check_random_state
import pandas as pd
from logging import getLogger
from pathlib import Path
import os
import sys
sys.path.append(os.pardir)

import conf
from visualization import plot_existing_new


# Set seeds to make the experiment reproducible
random_state = conf.random_state
random_ = check_random_state(random_state)

In [ ]:
logger = getLogger(__name__)
logger.info(f"The current working directory is {Path().cwd()}")

# log path
log_path = Path("../result/constrain")

df_path = log_path / "df"
df_path.mkdir(exist_ok=True, parents=True)

In [ ]:
x = "rho_L"
xticks = conf.rho_L_list
xticklabels = conf.rho_L_list
xlabels = r"lower limit on the proportion of new actions under the learned policy ($\rho_L$)"

In [ ]:
all_data = pd.read_csv(df_path / "all_data_results.csv")

In [ ]:
all_data_ = pd.DataFrame([], columns=["seed", "method", "V", "existing_action_ratio", "new_action_ratio", "existing_V", "new_V", "per_existing_V", "per_new_V", "estimated_V", "estimated_new_action_ratio"])

for seed in all_data["seed"].unique():
    lcpi = all_data[(all_data["method"] == "PONA (kappa=1.0)") & (all_data["seed"] == seed)]
    lcpi["method"] = "LCPI-PG"

    candidate_policy = all_data[(all_data["method"].str.contains("kappa")) & (all_data["seed"] == seed)]
    
    constrain_policy = candidate_policy.loc[[candidate_policy["estimated_V"].idxmax()]].copy()
    constrain_policy["method"] = "PONA (rho_L=-infty)"

    other_policy = all_data[(all_data["method"] != "PONA (varying rho_L)") & (all_data["seed"] == seed)]
    new_data = pd.concat([other_policy.reset_index(drop=True), constrain_policy.reset_index(drop=True), lcpi.reset_index(drop=True)], axis=0, ignore_index=True)
    all_data_ = pd.concat([all_data_, new_data], axis=0, ignore_index=True)
        

In [ ]:
all_data = all_data_

In [ ]:
all_rho_L_data = pd.DataFrame([], columns=["seed", x, "method", "V", "existing_action_ratio", "new_action_ratio", "existing_V", "new_V", "per_existing_V", "per_new_V", "estimated_V", "estimated_new_action_ratio"])

for rho_L in conf.rho_L_list:

    for seed in all_data["seed"].unique():

        candidate_policy = all_data[(all_data["method"].str.contains("kappa")) & (all_data["seed"] == seed)]
        candidate_policy = candidate_policy.query(f"estimated_new_action_ratio >= {rho_L}")

        if candidate_policy.empty:
            constrain_policy = all_data[(all_data["method"] == f"PONA (kappa={conf.tuning_kappa_list[-1]})") & (all_data["seed"] == seed)].copy()
        else:
            constrain_policy = candidate_policy.loc[[candidate_policy["estimated_V"].idxmax()]].copy()

        constrain_policy["method"] = "PONA (varying rho_L)"

        other_policy = all_data[(all_data["method"] != "PONA (varying rho_L)") & (all_data["seed"] == seed)]

        new_data = pd.concat([other_policy.reset_index(drop=True), constrain_policy.reset_index(drop=True)], axis=0, ignore_index=True)
        
        new_data["rho_L"] = rho_L

        if not all_rho_L_data.columns.equals(new_data.columns):
            new_data = new_data.reindex(columns=all_rho_L_data.columns)

        all_rho_L_data = pd.concat([all_rho_L_data, new_data], axis=0, ignore_index=True)

In [ ]:
conf.show_learner_list = ["PONA (rho_L=-infty)", "PONA (varying rho_L)", "PONA (kappa=1.0)"]

In [ ]:
plot_existing_new(
    all_rho_L_data,
    x, 
    xticks, 
    xticklabels, 
    xlabels, 
    fig_1="V", 
    fig_2="per_existing_V", 
    fig_3="per_new_V", 
    relative = conf.show_relative,
    # flag_legend = False,
)

In [ ]:
plot_existing_new(
    all_rho_L_data,
    x, 
    xticks, 
    xticklabels, 
    xlabels, 
    fig_1="V", 
    fig_2="existing_V", 
    fig_3="new_V", 
    relative = conf.show_relative,
    # flag_legend = False, 
)

In [ ]:
plot_existing_new(
    all_rho_L_data,
    x, 
    xticks, 
    xticklabels, 
    xlabels, 
    fig_1="V", 
    fig_2="existing_action_ratio", 
    fig_3="new_action_ratio", 
    relative = conf.show_relative,
    flag_legend = False, 
)

In [ ]:
import matplotlib.pyplot as plt
plt.style.use("ggplot")
fig, ax = plt.subplots(figsize=(10, 2)) 
show_learner_list = [r"PONA ($\rho_L=-\infty$)", r"PONA (varying $\rho_L$)", r"PONA ($\kappa=1.0$)"]
for estimator_name in show_learner_list:
    ax.plot([], [], color=conf.color_dict[estimator_name], marker='o', label=estimator_name, markersize=13, linewidth=5)

ax.legend(loc="center", ncol=len(show_learner_list), fontsize=18)
ax.axis('off')  

fig.subplots_adjust(left=0.8, right=0.9, top=0.2, bottom=0.1) 